In [1]:
"""
Summarizes average internal_distance / external_distance from trip_file_juris.csv
by (startTaz, endTaz, partyType, tripType, mode), and appends those averages as
new columns onto Trip_file_TAF_A1_2035.csv, matching on the same fields.

Note: Trip_file_TAF_A1_2035.csv stores 'partyType' and 'mode' as fixed-width
truncated strings (e.g. 'overnight visitor' -> 'overnight', 'drive to transit'
-> 'drive to transi'). This script truncates the juris file's values to match
before grouping/joining, so rows actually line up.
"""

import pandas as pd
import os

os.chdir("P:\\travelModel\\Scenarios_2026_04_28\\Analysis\\Post_TRIA_VMT")



In [2]:

# Alt 1 2035
# ---- file paths ----
JURIS_PATH_Alt1_2035 = '\\RawData\\trip_file_juris_Alt1_2035.csv'
TAF_PATH_Alt1_2035 = '\\RawData\\Trip_file_TAF_Alt1_2035.csv'
OUTPUT_PATH_Alt1_2035 = 'Trip_file_TAF_Alt1_2035_with_avg_distance.csv'

# Truncation lengths observed in the TAF file (fixed-width export artifact)
PARTYTYPE_MAXLEN = 10
MODE_MAXLEN = 15

JOIN_KEYS = ["startTaz", "endTaz", "partyType", "tripType", "mode"]  # juris names
TAF_KEYS = ["startTAZ", "endTAZ", "partyType", "tripType", "mode"]   # TAF names

# ---- 1. load the dataset to summarize ----
# robustly locate the juris CSV (handle leading slashes/backslashes making the path appear absolute)
_juris_path = JURIS_PATH_Alt1_2035
if _juris_path.startswith(("\\", "/")):
    candidate = os.path.normpath(os.path.join(os.getcwd(), _juris_path.lstrip("\\/")))
else:
    candidate = os.path.normpath(_juris_path)

# fallback: try converting backslashes to forward and treating as relative
if not os.path.exists(candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _juris_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        candidate = alt

if not os.path.exists(candidate):
    raise FileNotFoundError(f"Could not find juris file at '{JURIS_PATH_Alt1_2035}' (tried '{candidate}').")

juris = pd.read_csv(candidate, usecols=JOIN_KEYS + ["internal_distance", "external_distance"])

# Match the TAF file's truncated text formatting so the join keys align
juris["partyType"] = juris["partyType"].str.slice(0, PARTYTYPE_MAXLEN).str.strip()
juris["mode"] = juris["mode"].str.slice(0, MODE_MAXLEN).str.strip()

# ---- 2. compute the average distances per combination ----
summary = (
    juris.groupby(JOIN_KEYS, as_index=False)
    .agg(
        avg_internal_distance=("internal_distance", "mean"),
        avg_external_distance=("external_distance", "mean"),
    )
)

# Rename to match the TAF file's column names for the merge
summary = summary.rename(columns={"startTaz": "startTAZ", "endTaz": "endTAZ"})

# ---- 3. load the target dataset and merge the summary on as new columns ----
_taf_path = TAF_PATH_Alt1_2035
if _taf_path.startswith(("\\", "/")):
    taf_candidate = os.path.normpath(os.path.join(os.getcwd(), _taf_path.lstrip("\\/")))
else:
    taf_candidate = os.path.normpath(_taf_path)

if not os.path.exists(taf_candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _taf_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        taf_candidate = alt

if not os.path.exists(taf_candidate):
    raise FileNotFoundError(f"Could not find TAF file at '{TAF_PATH_Alt1_2035}' (tried '{taf_candidate}').")

taf = pd.read_csv(taf_candidate)

merged = taf.merge(summary, on=TAF_KEYS, how="left")

# ---- 4. save ----
_output_path = OUTPUT_PATH_Alt1_2035
if _output_path.startswith(("\\", "/")):
    output_candidate = os.path.normpath(
        os.path.join(os.getcwd(), _output_path.lstrip("\\/"))
    )
else:
    output_candidate = os.path.normpath(_output_path)

output_dir = os.path.dirname(output_candidate)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

merged.to_csv(output_candidate, index=False)

# ---- quick diagnostics ----
n_matched = merged["avg_internal_distance"].notna().sum()
print(f"TAF rows: {len(taf)}")
print(f"Rows with a matched average internal_distance: {n_matched} ({n_matched / len(taf):.1%})")
print(f"Unique combinations in juris summary: {len(summary)}")
print(f"Saved to: {OUTPUT_PATH_Alt1_2035}")


TAF rows: 98507
Rows with a matched average internal_distance: 86636 (87.9%)
Unique combinations in juris summary: 101340
Saved to: Trip_file_TAF_Alt1_2035_with_avg_distance.csv


In [3]:

# Alt 1 2050
# ---- file paths ----
JURIS_PATH_Alt1_2050 = "\\RawData\\trip_file_juris_Alt1_2050.csv"
TAF_PATH_Alt1_2050 = "\\RawData\\Trip_file_TAF_Alt1_2050.csv"
OUTPUT_PATH_Alt1_2050 = "Trip_file_TAF_Alt1_2050_with_avg_distance.csv"

# Truncation lengths observed in the TAF file (fixed-width export artifact)
PARTYTYPE_MAXLEN = 10
MODE_MAXLEN = 15

JOIN_KEYS = ["startTaz", "endTaz", "partyType", "tripType", "mode"]  # juris names
TAF_KEYS = ["startTAZ", "endTAZ", "partyType", "tripType", "mode"]   # TAF names

# ---- 1. load the dataset to summarize ----
_juris_path = JURIS_PATH_Alt1_2050
if _juris_path.startswith(("\\", "/")):
    candidate = os.path.normpath(os.path.join(os.getcwd(), _juris_path.lstrip("\\/")))
else:
    candidate = os.path.normpath(_juris_path)

# fallback: try converting backslashes to forward and treating as relative
if not os.path.exists(candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _juris_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        candidate = alt

if not os.path.exists(candidate):
    raise FileNotFoundError(f"Could not find juris file at '{JURIS_PATH_Alt1_2050}' (tried '{candidate}').")

juris = pd.read_csv(candidate, usecols=JOIN_KEYS + ["internal_distance", "external_distance"])


# Match the TAF file's truncated text formatting so the join keys align
juris["partyType"] = juris["partyType"].str.slice(0, PARTYTYPE_MAXLEN).str.strip()
juris["mode"] = juris["mode"].str.slice(0, MODE_MAXLEN).str.strip()

# ---- 2. compute the average distances per combination ----
summary = (
    juris.groupby(JOIN_KEYS, as_index=False)
    .agg(
        avg_internal_distance=("internal_distance", "mean"),
        avg_external_distance=("external_distance", "mean"),
    )
)

# Rename to match the TAF file's column names for the merge
summary = summary.rename(columns={"startTaz": "startTAZ", "endTaz": "endTAZ"})

# ---- 3. load the target dataset and merge the summary on as new columns ----
_taf_path = TAF_PATH_Alt1_2050
if _taf_path.startswith(("\\", "/")):
    taf_candidate = os.path.normpath(os.path.join(os.getcwd(), _taf_path.lstrip("\\/")))
else:
    taf_candidate = os.path.normpath(_taf_path)

if not os.path.exists(taf_candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _taf_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        taf_candidate = alt

if not os.path.exists(taf_candidate):
    raise FileNotFoundError(f"Could not find TAF file at '{TAF_PATH_Alt1_2050}' (tried '{taf_candidate}').")

taf = pd.read_csv(taf_candidate)

merged = taf.merge(summary, on=TAF_KEYS, how="left")

# ---- 4. save ----
_output_path = OUTPUT_PATH_Alt1_2050
if _output_path.startswith(("\\", "/")):
    output_candidate = os.path.normpath(
        os.path.join(os.getcwd(), _output_path.lstrip("\\/"))
    )
else:
    output_candidate = os.path.normpath(_output_path)

output_dir = os.path.dirname(output_candidate)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

merged.to_csv(output_candidate, index=False)


# ---- quick diagnostics ----
n_matched = merged["avg_internal_distance"].notna().sum()
print(f"TAF rows: {len(taf)}")
print(f"Rows with a matched average internal_distance: {n_matched} ({n_matched / len(taf):.1%})")
print(f"Unique combinations in juris summary: {len(summary)}")
print(f"Saved to: {OUTPUT_PATH_Alt1_2050}")


TAF rows: 101609
Rows with a matched average internal_distance: 89260 (87.8%)
Unique combinations in juris summary: 104647
Saved to: Trip_file_TAF_Alt1_2050_with_avg_distance.csv


In [ ]:

# Alt 2 2035
# ---- file paths ----
JURIS_PATH_Alt2_2035 = '\\RawData\\trip_file_juris_Alt2_2035.csv'
TAF_PATH_Alt2_2035 = '\\RawData\\Trip_file_TAF_Alt2_2035.csv'
OUTPUT_PATH_Alt2_2035 = 'Trip_file_TAF_Alt2_2035_with_avg_distance.csv'

# Truncation lengths observed in the TAF file (fixed-width export artifact)
PARTYTYPE_MAXLEN = 10
MODE_MAXLEN = 15

JOIN_KEYS = ["startTaz", "endTaz", "partyType", "tripType", "mode"]  # juris names
TAF_KEYS = ["startTAZ", "endTAZ", "partyType", "tripType", "mode"]   # TAF names

# ---- 1. load the dataset to summarize ----
# robustly locate the juris CSV (handle leading slashes/backslashes making the path appear absolute)
_juris_path = JURIS_PATH_Alt2_2035
if _juris_path.startswith(("\\", "/")):
    candidate = os.path.normpath(os.path.join(os.getcwd(), _juris_path.lstrip("\\/")))
else:
    candidate = os.path.normpath(_juris_path)

# fallback: try converting backslashes to forward and treating as relative
if not os.path.exists(candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _juris_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        candidate = alt

if not os.path.exists(candidate):
    raise FileNotFoundError(f"Could not find juris file at '{JURIS_PATH_Alt2_2035}' (tried '{candidate}').")

juris = pd.read_csv(candidate, usecols=JOIN_KEYS + ["internal_distance", "external_distance", "trip_time"])

# Match the TAF file's truncated text formatting so the join keys align
juris["partyType"] = juris["partyType"].str.slice(0, PARTYTYPE_MAXLEN).str.strip()
juris["mode"] = juris["mode"].str.slice(0, MODE_MAXLEN).str.strip()

# ---- 2. compute the average distances per combination ----
summary = (
    juris.groupby(JOIN_KEYS, as_index=False)
    .agg(
        avg_internal_distance=("internal_distance", "mean"),
        avg_external_distance=("external_distance", "mean"),
        avg_trip_time=("trip_time", "mean"),
    )
)

# Rename to match the TAF file's column names for the merge
summary = summary.rename(columns={"startTaz": "startTAZ", "endTaz": "endTAZ"})

# ---- 3. load the target dataset and merge the summary on as new columns ----
_taf_path = TAF_PATH_Alt2_2035
if _taf_path.startswith(("\\", "/")):
    taf_candidate = os.path.normpath(os.path.join(os.getcwd(), _taf_path.lstrip("\\/")))
else:
    taf_candidate = os.path.normpath(_taf_path)

if not os.path.exists(taf_candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _taf_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        taf_candidate = alt

if not os.path.exists(taf_candidate):
    raise FileNotFoundError(f"Could not find TAF file at '{TAF_PATH_Alt2_2035}' (tried '{taf_candidate}').")

taf = pd.read_csv(taf_candidate)

merged = taf.merge(summary, on=TAF_KEYS, how="left")

# ---- 4. save ----
_output_path = OUTPUT_PATH_Alt2_2035
if _output_path.startswith(("\\", "/")):
    output_candidate = os.path.normpath(
        os.path.join(os.getcwd(), _output_path.lstrip("\\/"))
    )
else:
    output_candidate = os.path.normpath(_output_path)

output_dir = os.path.dirname(output_candidate)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

merged.to_csv(output_candidate, index=False)

# ---- quick diagnostics ----
n_matched = merged["avg_internal_distance"].notna().sum()
print(f"TAF rows: {len(taf)}")
print(f"Rows with a matched average internal_distance: {n_matched} ({n_matched / len(taf):.1%})")
print(f"Unique combinations in juris summary: {len(summary)}")
print(f"Saved to: {OUTPUT_PATH_Alt2_2035}")


SyntaxError: '(' was never closed (714813632.py, line 38)

In [5]:

# Alt 2 2050
# ---- file paths ----
JURIS_PATH_Alt2_2050 = "\\RawData\\trip_file_juris_Alt2_2050.csv"
TAF_PATH_Alt2_2050 = "\\RawData\\Trip_file_TAF_Alt2_2050.csv"
OUTPUT_PATH_Alt2_2050 = "Trip_file_TAF_Alt2_2050_with_avg_distance.csv"

# Truncation lengths observed in the TAF file (fixed-width export artifact)
PARTYTYPE_MAXLEN = 10
MODE_MAXLEN = 15

JOIN_KEYS = ["startTaz", "endTaz", "partyType", "tripType", "mode"]  # juris names
TAF_KEYS = ["startTAZ", "endTAZ", "partyType", "tripType", "mode"]   # TAF names

# ---- 1. load the dataset to summarize ----
_juris_path = JURIS_PATH_Alt2_2050
if _juris_path.startswith(("\\", "/")):
    candidate = os.path.normpath(os.path.join(os.getcwd(), _juris_path.lstrip("\\/")))
else:
    candidate = os.path.normpath(_juris_path)

# fallback: try converting backslashes to forward and treating as relative
if not os.path.exists(candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _juris_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        candidate = alt

if not os.path.exists(candidate):
    raise FileNotFoundError(f"Could not find juris file at '{JURIS_PATH_Alt2_2050}' (tried '{candidate}').")

juris = pd.read_csv(candidate, usecols=JOIN_KEYS + ["internal_distance", "external_distance"])


# Match the TAF file's truncated text formatting so the join keys align
juris["partyType"] = juris["partyType"].str.slice(0, PARTYTYPE_MAXLEN).str.strip()
juris["mode"] = juris["mode"].str.slice(0, MODE_MAXLEN).str.strip()

# ---- 2. compute the average distances per combination ----
summary = (
    juris.groupby(JOIN_KEYS, as_index=False)
    .agg(
        avg_internal_distance=("internal_distance", "mean"),
        avg_external_distance=("external_distance", "mean"),
    )
)

# Rename to match the TAF file's column names for the merge
summary = summary.rename(columns={"startTaz": "startTAZ", "endTaz": "endTAZ"})

# ---- 3. load the target dataset and merge the summary on as new columns ----
_taf_path = TAF_PATH_Alt2_2050
if _taf_path.startswith(("\\", "/")):
    taf_candidate = os.path.normpath(os.path.join(os.getcwd(), _taf_path.lstrip("\\/")))
else:
    taf_candidate = os.path.normpath(_taf_path)

if not os.path.exists(taf_candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _taf_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        taf_candidate = alt

if not os.path.exists(taf_candidate):
    raise FileNotFoundError(f"Could not find TAF file at '{TAF_PATH_Alt2_2050}' (tried '{taf_candidate}').")

taf = pd.read_csv(taf_candidate)

merged = taf.merge(summary, on=TAF_KEYS, how="left")

# ---- 4. save ----
_output_path = OUTPUT_PATH_Alt2_2050
if _output_path.startswith(("\\", "/")):
    output_candidate = os.path.normpath(
        os.path.join(os.getcwd(), _output_path.lstrip("\\/"))
    )
else:
    output_candidate = os.path.normpath(_output_path)

output_dir = os.path.dirname(output_candidate)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

merged.to_csv(output_candidate, index=False)


# ---- quick diagnostics ----
n_matched = merged["avg_internal_distance"].notna().sum()
print(f"TAF rows: {len(taf)}")
print(f"Rows with a matched average internal_distance: {n_matched} ({n_matched / len(taf):.1%})")
print(f"Unique combinations in juris summary: {len(summary)}")
print(f"Saved to: {OUTPUT_PATH_Alt2_2050}")


TAF rows: 103597
Rows with a matched average internal_distance: 91123 (88.0%)
Unique combinations in juris summary: 106568
Saved to: Trip_file_TAF_Alt2_2050_with_avg_distance.csv


In [6]:

# Alt 3 2035
# ---- file paths ----
JURIS_PATH_Alt3_2035 = '\\RawData\\trip_file_juris_Alt3_2035.csv'
TAF_PATH_Alt3_2035 = '\\RawData\\Trip_file_TAF_Alt3_2035.csv'
OUTPUT_PATH_Alt3_2035 = 'Trip_file_TAF_Alt3_2035_with_avg_distance.csv'

# Truncation lengths observed in the TAF file (fixed-width export artifact)
PARTYTYPE_MAXLEN = 10
MODE_MAXLEN = 15

JOIN_KEYS = ["startTaz", "endTaz", "partyType", "tripType", "mode"]  # juris names
TAF_KEYS = ["startTAZ", "endTAZ", "partyType", "tripType", "mode"]   # TAF names

# ---- 1. load the dataset to summarize ----
# robustly locate the juris CSV (handle leading slashes/backslashes making the path appear absolute)
_juris_path = JURIS_PATH_Alt3_2035
if _juris_path.startswith(("\\", "/")):
    candidate = os.path.normpath(os.path.join(os.getcwd(), _juris_path.lstrip("\\/")))
else:
    candidate = os.path.normpath(_juris_path)

# fallback: try converting backslashes to forward and treating as relative
if not os.path.exists(candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _juris_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        candidate = alt

if not os.path.exists(candidate):
    raise FileNotFoundError(f"Could not find juris file at '{JURIS_PATH_Alt3_2035}' (tried '{candidate}').")

juris = pd.read_csv(candidate, usecols=JOIN_KEYS + ["internal_distance", "external_distance"])

# Match the TAF file's truncated text formatting so the join keys align
juris["partyType"] = juris["partyType"].str.slice(0, PARTYTYPE_MAXLEN).str.strip()
juris["mode"] = juris["mode"].str.slice(0, MODE_MAXLEN).str.strip()

# ---- 2. compute the average distances per combination ----
summary = (
    juris.groupby(JOIN_KEYS, as_index=False)
    .agg(
        avg_internal_distance=("internal_distance", "mean"),
        avg_external_distance=("external_distance", "mean"),
    )
)

# Rename to match the TAF file's column names for the merge
summary = summary.rename(columns={"startTaz": "startTAZ", "endTaz": "endTAZ"})

# ---- 3. load the target dataset and merge the summary on as new columns ----
_taf_path = TAF_PATH_Alt3_2035
if _taf_path.startswith(("\\", "/")):
    taf_candidate = os.path.normpath(os.path.join(os.getcwd(), _taf_path.lstrip("\\/")))
else:
    taf_candidate = os.path.normpath(_taf_path)

if not os.path.exists(taf_candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _taf_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        taf_candidate = alt

if not os.path.exists(taf_candidate):
    raise FileNotFoundError(f"Could not find TAF file at '{TAF_PATH_Alt3_2035}' (tried '{taf_candidate}').")

taf = pd.read_csv(taf_candidate)

merged = taf.merge(summary, on=TAF_KEYS, how="left")

# ---- 4. save ----
_output_path = OUTPUT_PATH_Alt3_2035
if _output_path.startswith(("\\", "/")):
    output_candidate = os.path.normpath(
        os.path.join(os.getcwd(), _output_path.lstrip("\\/"))
    )
else:
    output_candidate = os.path.normpath(_output_path)

output_dir = os.path.dirname(output_candidate)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

merged.to_csv(output_candidate, index=False)

# ---- quick diagnostics ----
n_matched = merged["avg_internal_distance"].notna().sum()
print(f"TAF rows: {len(taf)}")
print(f"Rows with a matched average internal_distance: {n_matched} ({n_matched / len(taf):.1%})")
print(f"Unique combinations in juris summary: {len(summary)}")
print(f"Saved to: {OUTPUT_PATH_Alt3_2035}")


TAF rows: 98605
Rows with a matched average internal_distance: 86860 (88.1%)
Unique combinations in juris summary: 101462
Saved to: Trip_file_TAF_Alt3_2035_with_avg_distance.csv


In [7]:

# Alt 3 2050
# ---- file paths ----
JURIS_PATH_Alt3_2050 = "\\RawData\\trip_file_juris_Alt3_2050.csv"
TAF_PATH_Alt3_2050 = "\\RawData\\Trip_file_TAF_Alt3_2050.csv"
OUTPUT_PATH_Alt3_2050 = "Trip_file_TAF_Alt3_2050_with_avg_distance.csv"

# Truncation lengths observed in the TAF file (fixed-width export artifact)
PARTYTYPE_MAXLEN = 10
MODE_MAXLEN = 15

JOIN_KEYS = ["startTaz", "endTaz", "partyType", "tripType", "mode"]  # juris names
TAF_KEYS = ["startTAZ", "endTAZ", "partyType", "tripType", "mode"]   # TAF names

# ---- 1. load the dataset to summarize ----
_juris_path = JURIS_PATH_Alt3_2050
if _juris_path.startswith(("\\", "/")):
    candidate = os.path.normpath(os.path.join(os.getcwd(), _juris_path.lstrip("\\/")))
else:
    candidate = os.path.normpath(_juris_path)

# fallback: try converting backslashes to forward and treating as relative
if not os.path.exists(candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _juris_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        candidate = alt

if not os.path.exists(candidate):
    raise FileNotFoundError(f"Could not find juris file at '{JURIS_PATH_Alt3_2050}' (tried '{candidate}').")

juris = pd.read_csv(candidate, usecols=JOIN_KEYS + ["internal_distance", "external_distance"])


# Match the TAF file's truncated text formatting so the join keys align
juris["partyType"] = juris["partyType"].str.slice(0, PARTYTYPE_MAXLEN).str.strip()
juris["mode"] = juris["mode"].str.slice(0, MODE_MAXLEN).str.strip()

# ---- 2. compute the average distances per combination ----
summary = (
    juris.groupby(JOIN_KEYS, as_index=False)
    .agg(
        avg_internal_distance=("internal_distance", "mean"),
        avg_external_distance=("external_distance", "mean"),
    )
)

# Rename to match the TAF file's column names for the merge
summary = summary.rename(columns={"startTaz": "startTAZ", "endTaz": "endTAZ"})

# ---- 3. load the target dataset and merge the summary on as new columns ----
_taf_path = TAF_PATH_Alt3_2050
if _taf_path.startswith(("\\", "/")):
    taf_candidate = os.path.normpath(os.path.join(os.getcwd(), _taf_path.lstrip("\\/")))
else:
    taf_candidate = os.path.normpath(_taf_path)

if not os.path.exists(taf_candidate):
    alt = os.path.normpath(os.path.join(os.getcwd(), _taf_path.replace("\\", "/").lstrip("/")))
    if os.path.exists(alt):
        taf_candidate = alt

if not os.path.exists(taf_candidate):
    raise FileNotFoundError(f"Could not find TAF file at '{TAF_PATH_Alt3_2050}' (tried '{taf_candidate}').")

taf = pd.read_csv(taf_candidate)

merged = taf.merge(summary, on=TAF_KEYS, how="left")

# ---- 4. save ----
_output_path = OUTPUT_PATH_Alt3_2050
if _output_path.startswith(("\\", "/")):
    output_candidate = os.path.normpath(
        os.path.join(os.getcwd(), _output_path.lstrip("\\/"))
    )
else:
    output_candidate = os.path.normpath(_output_path)

output_dir = os.path.dirname(output_candidate)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

merged.to_csv(output_candidate, index=False)


# ---- quick diagnostics ----
n_matched = merged["avg_internal_distance"].notna().sum()
print(f"TAF rows: {len(taf)}")
print(f"Rows with a matched average internal_distance: {n_matched} ({n_matched / len(taf):.1%})")
print(f"Unique combinations in juris summary: {len(summary)}")
print(f"Saved to: {OUTPUT_PATH_Alt3_2050}")

TAF rows: 102652
Rows with a matched average internal_distance: 90470 (88.1%)
Unique combinations in juris summary: 105552
Saved to: Trip_file_TAF_Alt3_2050_with_avg_distance.csv
